# RLRLLL's column sweeper is a checked macro-transition

## Question
After ~1.27e10 steps from a blank board, `RLRLLL` stops growing as a blob and sweeps up and down, laying a period-3 wake and
moving west. Simulation can only show that this continues for a finite time. Is there a finite family of configurations
`C(m, q)` (bulk height `m`, phase `q`) that the rule maps into itself, `C(m, q) -> C(m + g, q + 1)`, for **every** `m`?
If so, the sweeper is proved to continue forever and its asymptotics become exact.

## Assumptions and definitions
Blank board, ant at the centre heading north, update order of `src/reference.rs`. Front = minimum visited x; a cycle ends
each time the front has moved 3 more columns west. Predictions P1-P5 were registered in the README (commit `4bc029a`)
before any run. The proof claim is judged by a separate checker (`examples/rlrlll_macro_transition_check.rs`) whose
every concrete step is a `reference::Ant` step; a symbolic band of identical rows is crossed only by a *shift lemma*,
a finite run on the band's block plus a few neighbouring rows that must reproduce itself one block further on.

In [1]:
import hashlib, subprocess
from pathlib import Path
import numpy as np
import polars as pl
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
R = ROOT / "experiments/archive/2026-10-rlrlll-macro-transition/results"
for n in ["cycles.csv", "cycles_early.csv", "canonical_k0.txt", "check_k0.txt", "check_negative_controls.txt"]:
    print(hashlib.sha256((R / n).read_bytes()).hexdigest()[:16], n)
print("git", subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip())

b74e68ae40e921e4 cycles.csv
249ef7f989be4545 cycles_early.csv
e6aba0ba81fa46c4 canonical_k0.txt
017cf115a45fe366 check_k0.txt
eadf169861297f2c check_negative_controls.txt
git 35d8e27


## Stage 1: the measured cycle law (P1-P3, P5)

In [2]:
d = pl.read_csv(R / "cycles.csv")
T, L, reach = d["steps"].to_numpy(), d["L_prev_col"].to_numpy(), d["reach"].to_numpy()
beta = T - 12 * L
print("cycles:", d.height)
print("T - 12 L values:", dict(zip(*np.unique(beta, return_counts=True))), " period 3:", bool(np.all(beta[3:] == beta[:-3])))
print("growth L_(k+1) - L_k values:", dict(zip(*np.unique(np.diff(L), return_counts=True))))
print("max reach behind the front:", reach.max())
ax = np.abs(d["x_f"].to_numpy()).astype(float)
slope = np.diff(ax**2) / T[1:]
for idx in np.array_split(np.arange(len(slope)), 5):
    print(f"  |x| ~ {ax[1:][idx].mean():7.0f}: d|x|^2/dt = {slope[idx].mean():.4f}")

cycles: 4729
T - 12 L values: {np.int64(358): np.int64(1577), np.int64(394): np.int64(1576), np.int64(518): np.int64(1576)}  period 3: True
growth L_(k+1) - L_k values: {np.int64(2): np.int64(4728)}
max reach behind the front: 20
  |x| ~    3618: d|x|^2/dt = 1.3326
  |x| ~    6456: d|x|^2/dt = 0.9689
  |x| ~    9294: d|x|^2/dt = 0.8876
  |x| ~   12130: d|x|^2/dt = 0.8506
  |x| ~   14965: d|x|^2/dt = 0.8294


Every cycle obeys `T = 12 L + beta` exactly with `beta` cycling through three values; every cycle adds exactly 2 rows;
the ant never reads more than 20 columns behind the front. The local slope of `|x|^2` against `t` is still falling at
the board edge: the earlier fit of 0.8634 was not an asymptote.

## The proposed family
Read off snapshots 0-2 (cells: colour digits, `.` never visited; the ant is at the front column, 6 or 7 rows below the
top). Only phase 0 is shown, cropped to 26 columns.

In [3]:
txt = (R / "canonical_k0.txt").read_text().splitlines()
start = next(i for i, l in enumerate(txt) if l.startswith("phase 0"))
for l in txt[start:start + 50]:
    print(l[:26] if not l.startswith(("phase", "top", "bulk", "bottom")) else l)

phase 0 k 0 m 434 ant 0 6 3 width 48
top 29
.........11...............
........1331111111..1111..
.......143345544441133441.
..11..13555055444440550442
.1441105534300444435440443
.2344040005345344453035044
.1.22222105305305555055044
..104534243505354403404044
.1400530122343553542255544
.1444045300122340432253505
.1444003505300122344445354
.1444003545505300122503553
.1443503435545505300122340
.1443505123435545505300122
.1443543125123435545505300
.1443543053125123435545505
.1443543543053125123435545
.1443543543543053125123435
.1443543543543543053125123
.1443543543543543543053125
.1443543543543543543543053
.1443543543543543543543543
.1443543543543543543543543
.1443543543543543543543543
.1443543543543543543543543
.1443543543543543543543543
.1443543543543543543543543
.1443543543543543543543543
.1443543543543543543543543
bulk
.1443543543543543543543543
bottom 17
.1443543543543543543543543
.1443543543543543543543543
.1443543543543543543543543
.1443543543543543543543543
.1443543543543543543543

## Stage 2: the checker's verdict

In [4]:
print((R / "check_k0.txt").read_text())

# cargo run --release --example rlrlll_macro_transition_check -- RLRLLL experiments/2026-10-20-rlrlll-macro-transition/results/canonical_k0.txt
# binary sha256 9b57855e65713b07412859b29b9e3c6a35a1b18f71b4722b74652c6cba170494
phase 0 -> 1:
  lemma down H=2 p=2 s=20: band of (1 j + -1) blocks crossed
  lemma up   H=1 p=2 s=4: band of (1 j + -1) blocks crossed
  OK: C(m, 0) -> C(m + 2, 1) in 24/2 m + 934 steps, for all even m >= 2; ant (0, 7, 3)
  (proposal had m = 434 for this phase; next phase m = 436)
phase 1 -> 2:
  lemma down H=2 p=2 s=20: band of (1 j + -1) blocks crossed
  lemma up   H=1 p=2 s=4: band of (1 j + -1) blocks crossed
  OK: C(m, 1) -> C(m + 2, 2) in 24/2 m + 1058 steps, for all even m >= 2; ant (0, 6, 3)
  (proposal had m = 436 for this phase; next phase m = 438)
phase 2 -> 0:
  lemma down H=2 p=2 s=20: band of (1 j + -1) blocks crossed
  lemma up   H=1 p=2 s=4: band of (1 j + -1) blocks crossed
  OK: C(m, 2) -> C(m + 2, 0) in 24/2 m + 898 steps, for all even m >= 2; an

In [5]:
print((R / "check_negative_controls.txt").read_text())

# negative controls: mutated bulk cell (col 46: 3->4), and wrong rule RLRLLR
  FAILED: result is not C(m', 1): segments differ
  FAILED: result is not C(m', 2): segments differ
  FAILED: result is not C(m', 0): segments differ
  concrete check phase 0, m = 2: 958 steps, result MISMATCH
  concrete check phase 0, m = 4: 982 steps, result MISMATCH
  concrete check phase 0, m = 6: 1006 steps, result MISMATCH
  concrete check phase 0, m = 8: 1030 steps, result MISMATCH
  concrete check phase 0, m = 10: 1054 steps, result MISMATCH
## RLRLLR
  FAILED: left the strip at column 52
  FAILED: left the strip at column 52
  FAILED: result is not C(m', 0): segment counts differ: 50 vs 47
NOT CLOSED



For each phase, `C(m, q)` maps to `C(m + 2, q + 1)` in `12 m + b_q` steps for every even `m >= 2`. Two lemmas carry the
induction over the bulk: down at 20 steps per 2 rows, up at 4 steps per 2 rows, so 12 steps per bulk row. The 27
brute-force runs agree, and both negative controls are rejected (one changed bulk cell, and the wrong rule).

## Consequence for the actual run

In [6]:
m0 = 434
b = np.array([934, 1058, 898])
k = np.arange(len(T) - 1)
pred = 12 * (m0 + 2 * k) + b[k % 3]
print("measured cycles reproduced exactly:", int((T[1:] == pred).sum()), "of", len(k))
e = pl.read_csv(R / "cycles_early.csv")
print("early probe: first cycle at front x =", e["x_f"][0], "with column height", e["L_prev_col"][0],
      "; T - 12 L values", sorted(set((e["steps"] - 12 * e["L_prev_col"]).to_list())))

measured cycles reproduced exactly: 4728 of 4728
early probe: first cycle at front x = -1823 with column height 226 ; T - 12 L values [374, 402, 530]


## Entry probe (2026-10-21; registered in the README before measuring, but committed together with the results)
`results/entry_backtrack.txt` is the ant's turnaround log (swing >= 40 rows) walked backwards from the first front advance.
Registered outcome: E1a failed (column counts include the old blob) and E1b failed on its swing = L(x) clause, so E1 is
rejected as registered. Post hoc: the ant is on the sweeper clock well before the edge.

In [7]:
print(chr(10).join(l for l in (R / "entry_backtrack.txt").read_text().splitlines()
                 if l.startswith(("maxima", "run starts", "all cycles", "swing increments"))))

maxima: 67155 contiguous law-like run (max-turnaround x steps west 2..4, top rises 0..2 per cycle) from index 67061
run starts at t = 12676233468 x = -1536 y = -912  steps before T_END: 188299 cycles: 94
all cycles: T - 12 swing counts: Counter({358: 31, 374: 31, 514: 31})
swing increments per cycle: Counter({3: 31, 1: 31, 2: 30, 0: 1})


## Task A: cell-level family membership (2026-10-21; registration committed first as f67d3f4)
Windows of 48 columns at the ant's own front (minimum x since step 12,676,233,000), x = 2 (mod 3), compared with `C(m(x), q(x))`
from `canonical_k0.txt`, `m(x) = 434 + 2(-2197 - x)/3`. Controls (events at -2197..-2203 reproduce `cycles.csv`, match
C(434..438, 0..2) exactly, wrong (q, m) and one-cell perturbations fail) all pass.

In [8]:
print(chr(10).join((R / "window_controls.txt").read_text().splitlines()[:2] + (R / "window_controls.txt").read_text().splitlines()[-1:]))
print(chr(10).join(l for l in (R / "window_scan.txt").read_text().splitlines() if l.startswith(("A1", "A2", "A3", "A4", "primary", "secondary"))))
print(chr(10).join(l for l in (R / "window_cone.txt").read_text().splitlines() if l.startswith(("rect", "-1549", "-1552", "-1600"))))

Reading inline script metadata from `experiments/2026-10-20-rlrlll-macro-transition/scripts/window_compare.py`
C1 x=-2197 t=12677007177 (cycles.csv 12677007177: equal) primary=True flag=True ant_heading_ok=True n_in=0 n_out=0
CONTROLS ALL PASS
A1 clock law on 51 pairs with x <= -1552: all hold = True
A2 targets x <= -1594: 36, failing primary: [-1594, -1597, -1600, -1603, -1606, -1609, -1612, -1615, -1618, -1621, -1624, -1627, -1630, -1633, -1636, -1639, -1642, -1645, -1648, -1651, -1654, -1657, -1660, -1663, -1666, -1669, -1672, -1675, -1678, -1681, -1684, -1687, -1690, -1693, -1696, -1699]
A3 targets -1591..-1552: 14, passing primary: []; diff cells outside the two named sources (other, unknown_old) per target: {}
A4 (exploratory) 21-column window: eastmost failing target: -1549 ; passing 21-col targets: 2 of 51
primary 48-col: eastmost passing target None ; passing 0 of 51
secondary (visited flag too): passing 0 of 51
-1549 m=   2 q=0 rect_colour=False rect_flag=False cone_violation

Scored as registered: A1 (clock) held on 51 of 51 pairs; A2 **failed** (0 of 36 pass the registered whole-apron test, because
old blob cells lie outside the rectangle); A3 held, for a different reason than predicted. Post hoc (labelled as such): the
rectangle itself, ant offset and heading match exactly (colour and visited flag, zero nonzero cells in a 3-row light cone out to
320 columns) at 50 of 51 targets, every x <= -1552 (m >= 4); x = -1549 (m = 2) fails on 17 old blob cells inside the rectangle.
**The sweeper is on the closed family from step 12,676,237,591 (C(4, 1), x = -1552), 769,586 steps before the original base
case and 184,176 before the front crosses the blob edge.**

## Task B: below the family (registered before the run)
Same tool from step 12,676,223,500, targets x = -1471 .. -1537.

In [9]:
print((R / "window_seed.txt").read_text())

Reading inline script metadata from `experiments/2026-10-20-rlrlll-macro-transition/scripts/window_seed.py`
B1: 21 of 23 targets fired; skipped [-1471, -1474] (2)
   t strictly increasing with decreasing x: True
x     m    q  dt     formula  diff   L(new, col x+3)  dt-12L
-1477  -46 0      70    382   -312   n=  0 L=  0      70
-1480  -44 1      10    530   -520   n=  0 L=  0      10
-1483  -42 2     894    394    500   n=  0 L=  0      894
-1486  -40 0     370    454    -84   n=  1 L=  1      358
-1489  -38 1       6    602   -596   n=  4 L=  4      -42
-1492  -36 2     162    466   -304   n=  0 L=  0      162
-1495  -34 0      30    526   -496   n=  0 L=  0      30
-1498  -32 1      22    674   -652   n=  0 L=  0      22
-1501  -30 2      10    538   -528   n=  0 L=  0      10
-1504  -28 0      82    598   -516   n=  0 L=  0      82
-1507  -26 1     842    746     96   n=  3 L=  3      806
-1510  -24 2     610    610      0   n= 19 L= 19      382
-1513  -22 0     670    670      0   

B1 held (21 of 23 targets fire, monotone). **B2 failed**: the clock formula `12 m + c_q`, evaluated at negative `m`, holds exactly
on 9 pairs with m <= -6, down to x = -1510 (m = -24, column height 19, step 12,676,226,047) and fails at x = -1507. The column
heights are 3 (x = -1504), 19 (-1507), 21 (-1510), 23 (-1513): the wedge starts abruptly at height ~19, not at zero, so the
backward extrapolation to an apex at x ~ -1474 was wrong. The "swing 41" floor was my zigzag filter. B3 (sd of dt - 12 L > 10) is
numerically true but uninformative because the pooled pairs include the exact ones. B4 was not run. No cell-level statement is
made for m < 4: the family is undefined there.

Snapshot 0 is `C(434, 0)`, so given that one 1.27e10-step simulation, the blank-board `RLRLLL` ant never builds a
highway: it sweeps forever, with `|x|^2 / t -> 9/12 = 3/4`.

## Task C: assurance (2026-10-21; registered in the README first, with a dated correction of two registration errors)
**C1, a second checker.** `scripts/c1b.py` is written from the rule definition and the proposal text only (no `src/`, none of the first
checker's code), with its own ant (`c1_ant.py`) and a different proof architecture: five explicit finite runs (top block -> descent frame,
descent lemma^n, bottom turnaround, return lemma^n, top block -> new front) composed by row bookkeeping. `c1a.py` compares its ant with
`reference::Ant` through a thin Rust driver (a per-step hash of position, colour read and heading).

In [10]:
for n in ["c1a_conventions.txt", "c1a_semantics.txt", "c1b_verify.txt"]:
    print("== " + n)
    print(chr(10).join(l[:230] for l in (R / n).read_text().splitlines()))

== c1a_conventions.txt
table=CW R=+1: closes 12/12 of (q in 0..2) x (m in 2,4,10,50)
table=CW R=-1: closes 0/12 of (q in 0..2) x (m in 2,4,10,50)
table=CCW R=+1: closes 0/12 of (q in 0..2) x (m in 2,4,10,50)
table=CCW R=-1: closes 12/12 of (q in 0..2) x (m in 2,4,10,50)
closing candidates: [('CW', 1), ('CCW', -1)]
== c1a_semantics.txt
# C1a: independent ant vs reference::Ant; per-step FNV-1a fold of (x, y, colour read, heading after turn)
# driver sha256 in results/c1_driver.sha256; python 3.12.3
blank board 1000000 steps, ant rule RLRLLL (correct), reference rule RLRLLL: 1000 checkpoints; IDENTICAL
   final: steps 1000000 pos (-4, -22) heading 2 ncells 12699 cellhash a606cfd8f7d59482
blank board 1000000 steps, ant rule control 1a: colour 3 L->R, RLRRLL, reference rule RLRLLL: 1000 checkpoints; DISAGREE: first differing checkpoint at step 1000: 15b1ee54ff3dc167 vs 641e2e97d2b8a65e; final pos: mine -38 62 vs ref
   final: steps 1000000 pos (-38, 62) heading 0 ncells 19358 cellhash 41d92

The conventions test finds that two of four heading conventions close the cycle (the same ant under two labelings) and the mirror ants close
none. My ant agrees with `reference::Ant` at every one of 1,000 checkpoints of a 1e6-step blank-board run and on 12 window cycles; two
deliberately wrong rules (`RLRRLL`, `LLRLLL`) disagree from the first checkpoint. The second checker reproduces the first one's constants
exactly: `12 m + 934 / 1058 / 898`, descent lemma 20 steps per 2 rows, return lemma 4 steps per 2 rows, valid for every even `m >= 2`, and
150 brute-force cycles (m = 2..100, three phases) agree. It uses smaller frames (H = 1) because it cuts at the ant's arrival on a row.

In [11]:
for n in ["c1b_control_mut46.txt", "c1b_control_rlrllr.txt", "c1_controls.txt"]:
    print("== " + n)
    print(chr(10).join(l[:200] for l in (R / n).read_text().splitlines() if not l.startswith("  S5 with")))

== c1b_control_mut46.txt
# c1b: independent verifier. rule RLRLLL, canonical experiments/2026-10-20-rlrlll-macro-transition/results/canonical_k0_mut46.txt, python 3.12.3
phase 0 -> 1:
  S1: K1 = 1 band rows, 387 steps, ant arrives on row 44 at x = -1, heading 2; rows touched 15..44 of 0..43, x -2..9
  descent lemma: H = 1, p = 2, s = 20 steps per application; A = rows 43..43 of the S1 output; patch rows touched 0..3 of 0..2, x -2..1
  S3: e = 1 band rows, 216 steps; rows touched -1..19 of 0..32 (A = rows 0..0, bottom_q = rows 2..18), x -2..2; ant at x = 1, heading 0
  return lemma: H' = 1, p' = 2, s = 4 steps per application; patch rows touched 0..2 of 0..3, x 1..2
  S5: k5 = 0 band rows, 355 steps; rows touched 12..42 of 0..43, x -3..18; final ant x = -3, heading 3
  FAILED: composition: n = 0 (m = 2): final state is not C(m + 2, 1); 8 cells differ, first at window (x, y) = (46, 42)
phase 1 -> 2:
  S1: K1 = 1 band rows, 383 steps, ant arrives on row 45 at x = -1, heading 2; rows touch

Controls: a bulk cell changed in a column the ant never reads (trajectories identical) is rejected in all three phases at the composition
equality, like the first checker. `RLRLLR` is rejected in all three phases, but at a different place (the finite top-block run leaves its patch)
than the first checker's "left the strip" / "segment counts differ"; I score that control as split, not held (README).

**C2, the cone margin.** Per-cycle reach of the family's ant, from the finite runs and from 150 brute-force cycles, then the exact base-case
criterion: the real configuration at each of Task A's 50 events must equal the ideal initial configuration on every cell the ideal ant ever
stands on.

In [12]:
print((R / "c2_reach.txt").read_text())
print(chr(10).join(l[:330] for l in (R / "c2_recheck.txt").read_text().splitlines()[:4] + (R / "c2_recheck.txt").read_text().splitlines()[-2:]))
print((R / "c2_control.txt").read_text())

# (a) brute force, my ant; coordinates: rectangle top row y = 0, bottom row y = T + m + Bt - 1, front column x = 0
per phase max over m = 2..100 of rows visited above the top row / below the bottom row / max column visited:
  q=0: above 2, below 1, max column 18
  q=1: above 0, below 1, max column 17
  q=2: above 1, below 1, max column 10
overall: above (2, (0, 2)), below (1, (0, 2)), east (18, (0, 2))  (value, (q, m) first attaining it)
C2b: in all 150 cycles column -3 is visited only by the final position (and nothing further west): True
visited cells outside the rectangle rows that end at colour 0 (visited 6 times): 0; max visits to any such cell: 4
# (b) symbolic: finite runs of c1b.verify_phase (independent of m)
  q=0: S1 above 0, S5 above 2, S3 below 1, S1/S3/S5 max column 9/2/18 (lemma runs: descent x -2..1, return x 1..2); S1 min column -2, S3 -2, S5 -3
  q=1: S1 above 0, S5 above 0, S3 below 1, S1/S3/S5 max column 8/2/17 (lemma runs: descent x -2..1, return x 1..2); S1 min co

Registered C2a (at most 1 row per side) **failed**: a cycle visits up to 2 rows above its rectangle (phase 0). The rectangle's top moves up
2, 0, 1 rows per phase and its bottom down 1 per phase, so the wedge is 1 row per cycle per side. The exact criterion passes in all 50
windows (about 3e4 to 4.5e4 visited cells each, no mismatch), which confirms the base case without any observed margin. But **Task A's cone
(3 + s//3 rows) was too small**: every window has visited cells outside it, by up to 7 rows above, because a cycle's rows are visited up to 18
columns east of its front. Task A's cone test was not a sufficient certificate; the exact criterion is. One premise is not re-derived
here: nothing the real run visited lies west of x = -1820 (checked in the dumps for rows -2000..-1 and the apron, taken from the tracer's front
record elsewhere).

**C3, Lean 4.** `lean/` (core library only, no Mathlib): a list-of-rows model of the ant, a frame lemma (a run inside a patch extends to any
surrounding rows), and the two shift lemmas as iteration theorems for all `n`, instantiated for q = 0, 1, 2 from six finite certificates checked by
`decide +kernel`. No `sorry`; axioms propext and Quot.sound only.

In [13]:
print((R / "c3_axioms.txt").read_text())
print(chr(10).join(l for l in (R / "c3_build.txt").read_text().splitlines() if l.startswith(("Build", "BUILD", "Lean", "#"))))

'Sweeper.runN_emb' depends on axioms: [propext, Quot.sound]
'Sweeper.descent_iter' depends on axioms: [propext, Quot.sound]
'Sweeper.return_iter' depends on axioms: [propext, Quot.sound]
'Sweeper.descent_cert0' depends on axioms: [propext]
'Sweeper.descent_cert1' depends on axioms: [propext]
'Sweeper.descent_cert2' depends on axioms: [propext]
'Sweeper.return_cert0' depends on axioms: [propext]
'Sweeper.return_cert1' depends on axioms: [propext]
'Sweeper.return_cert2' depends on axioms: [propext]
'Sweeper.descent_all0' depends on axioms: [propext, Quot.sound]
'Sweeper.descent_all1' depends on axioms: [propext, Quot.sound]
'Sweeper.descent_all2' depends on axioms: [propext, Quot.sound]
'Sweeper.return_all0' depends on axioms: [propext, Quot.sound]
'Sweeper.return_all1' depends on axioms: [propext, Quot.sound]
'Sweeper.return_all2' depends on axioms: [propext, Quot.sound]

# clean rebuild: lake build (Lean 4.35.0 core, no Mathlib, no native_decide)
Lean (version 4.35.0, x86_64-unknown-li

**L4 continuation, 2026-10-22.** The earlier C3 log above is preserved. The end runs and composition are now checked too,
at implementation `f90733c412c95029da2eafeaa8923a69339bc2a6`. `Bounded.lean` checks every pre-step x in [-2,47], while permitting
the final arrival at -3. `IterB.lean` proves the bounded band inductions. `L4Data.lean` contains nine kernel-decided S1/S3/S5
certificates (387/383/383, 216 each, 355/483/323 steps) and six bounded descent/return certificates. `Cycle.lean` composes the five
stages for all n; `cycle0/1/2` and `cycle_runN0/1/2` prove the clocks 12m + 934/1058/898 for every m = 2+2n.
`composition0/1/2` prove the next-phase grid identity for all n by list algebra, and `first_arrival0/1/2` plus `target_ant0/1/2`
give first arrival at column -3 and the next ant state. These are single-cycle theorems in the finite list-of-rows model.

Reproduce: `bash experiments/archive/2026-10-rlrlll-macro-transition/scripts/l4_check.sh` under WSL, with the uv environment in the README.
The script regenerates proposed rows, uses a fresh build directory, and records hashes and the axiom audit. No new measurements or seeds.

In [14]:
for name in ["c3_l4_build.txt", "c3_l4_axioms.txt"]:
    raw = (R / name).read_bytes()
    print(name, "sha256", hashlib.sha256(raw).hexdigest())
    lines = raw.decode().splitlines()
    if name.endswith("build.txt"):
        print(chr(10).join(lines[:4] + lines[-3:]))
    else:
        assert "sorryAx" not in raw.decode() and "Lean.ofReduceBool" not in raw.decode()
        print(chr(10).join(line for line in lines if any(
            key in line for key in ["s1_cert", "s3_cert", "s5_cert", "composition", "cycle0", "cycle1", "cycle2", "first_arrival", "target_ant"])))

c3_l4_build.txt sha256 7aa4c2e566f3bf5bd154f8745e6cb8a1d73b7524f5a1bb18a6b884e866502b6f
# Command: bash experiments/2026-10-20-rlrlll-macro-transition/scripts/l4_check.sh
# Source commit: f90733c412c95029da2eafeaa8923a69339bc2a6; working source hashes below
# Deterministic finite certificates; no randomness; n universally quantified.
e6aba0ba81fa46c4ecf8e416ec32c7c9a8c854eae142c9cac1773a50ad1a8dc5  /mnt/f/Code/turmite/experiments/2026-10-20-rlrlll-macro-transition/results/canonical_k0.txt
✔ [8/10] Built Cycle (2.4s)
✔ [9/10] Built L4Data (32s)
Build completed successfully (10 jobs).
c3_l4_axioms.txt sha256 9e47047e7da49bd6dd86e46a687e2f8f706b8d263339b0e725b5e4aaa64f2ffc
'Sweeper.s1_cert0' depends on axioms: [propext]
'Sweeper.s1_cert1' depends on axioms: [propext]
'Sweeper.s1_cert2' depends on axioms: [propext]
'Sweeper.s3_cert0' depends on axioms: [propext]
'Sweeper.s3_cert1' depends on axioms: [propext]
'Sweeper.s3_cert2' depends on axioms: [propext]
'Sweeper.s5_cert0' depends on axi

**Multi-cycle padding remains open.** The output comparison has 12/14/13 blank rows above the target rectangle and 13 below,
where the next input asks for 14 on both sides. Shifting/cutting the finite rows and adding new western blank context is a defined
comparison operation, not a proved infinite-board extension. No Lean proof of padding replenishment or indefinitely many cycles exists here.

## What this does not show (caveats)
- **Entry.** The cell-level family holds from step 12,676,237,591 (x = -1552, m = 4); the clock formula extends exactly to x = -1510 (m = -24, column height 19, step 12,676,226,047); before that the log is irregular. The wedge starts at height ~19-21, not zero (the apex extrapolation was wrong). Not shown: what inside the blob corner produces that first column, and a cell-level description for m < 4. The old-blob cells outside the rectangle are excluded by a 3-row light-cone margin chosen from the observed growth, not derived from the checker. E1 (registered) was rejected as scored; A2 and B2 also failed as registered.
- **Trust.** The base case is one run of an uncertified plain loop (consistent with an independent run's exit step); the second checker
  did not re-run it. Two checkers (Rust, Python) by one author agree on every constant. Lean now checks bounded single cycles and composition
  for all `n`; infinite-plane lifting, multi-cycle padding and a model-correspondence theorem to Rust remain open.
- **Explanation.** The check certifies the period-3 wake and the bulk row `14435435...`; it does not say why they arise.
- P4 was tested on 373 sampled boundaries, not all 4,729 (the checker then covers all of them).

## Next
Lean cycle-to-cycle padding and infinite-plane lifting; the same propose-then-check loop on the other
no-highway escapes and on census highways (a "sweeper decider"); an explanation of the clock at m < 2, and what at the NW blob corner seeds the first column.